<a href="https://colab.research.google.com/github/lianghuizi/Study/blob/main/mim_rae_dit_experiment2_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/bytetriper/RAE.git
!git clone https://github.com/lianghuizi/RAE-DiT-Adapters.git
!cp RAE-DiT-Adapters/rae_latent_adapters.py RAE/src/stage2/models/

fatal: destination path 'RAE' already exists and is not an empty directory.
fatal: destination path 'RAE-DiT-Adapters' already exists and is not an empty directory.


In [2]:
%cd /content/RAE
!pip install -q -r requirements.txt
!pip install -q --no-cache-dir --force-reinstall "numpy==1.26.4"
!pip install -q wandb huggingface_hub omegaconf

/content/RAE
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.0 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.0 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.0 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.0 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have

In [3]:
%cd /content/RAE
!python /content/RAE-DiT-Adapters/make_configs.py \
  --rae-root /content/RAE \
  --epochs 50 \
  --global-batch-size 128 \
  --num-workers 0

/content/RAE
baseline: /content/RAE/configs/stage2/training/ImageNet256_adapter/DiTDH-S_DINOv2-B_baseline.yaml
channel_se: /content/RAE/configs/stage2/training/ImageNet256_adapter/DiTDH-S_DINOv2-B_channel_se.yaml
mim4: /content/RAE/configs/stage2/training/ImageNet256_adapter/DiTDH-S_DINOv2-B_mim4.yaml


In [4]:
!ls configs/stage2/training/ImageNet256_adapter

DiTDH-S_DINOv2-B_baseline.yaml	  DiTDH-S_DINOv2-B_mim4.yaml
DiTDH-S_DINOv2-B_channel_se.yaml


In [5]:
%cd /content/RAE
!pip install -q huggingface_hub

!huggingface-cli download nyu-visionx/RAE-collections \
  decoders/dinov2/wReg_base/ViTXL_n08/model.pt \
  stats/dinov2/wReg_base/imagenet1k/stat.pt \
  --local-dir models

/content/RAE
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 2 files: 100% 2/2 [00:00<00:00, 1089.15it/s]
/content/RAE/models


In [6]:
!ls -lh models/decoders/dinov2/wReg_base/ViTXL_n08/model.pt
!ls -lh models/stats/dinov2/wReg_base/imagenet1k/stat.pt

-rw-r--r-- 1 root root 1.6G Jun  6 06:13 models/decoders/dinov2/wReg_base/ViTXL_n08/model.pt
-rw-r--r-- 1 root root 770K Jun  6 06:13 models/stats/dinov2/wReg_base/imagenet1k/stat.pt


In [7]:
import shutil
from pathlib import Path
from torchvision import datasets
from PIL import Image
from tqdm.auto import tqdm

root = Path("/content/imagenet_like/train")

if root.exists():
    shutil.rmtree(root)
root.mkdir(parents=True, exist_ok=True)

cifar = datasets.CIFAR10(root="/content/data", train=True, download=True)
counts = {i: 0 for i in range(10)}

for img, label in tqdm(cifar, desc="export full CIFAR-10"):
    cls = root / f"class_{label:03d}"
    cls.mkdir(parents=True, exist_ok=True)
    img = img.resize((256, 256), Image.BICUBIC)
    img.save(cls / f"{counts[label]:05d}.png")
    counts[label] += 1

print(counts)
print("total:", sum(counts.values()))

100%|██████████| 170M/170M [00:16<00:00, 10.1MB/s]


export full CIFAR-10:   0%|          | 0/50000 [00:00<?, ?it/s]

{0: 5000, 1: 5000, 2: 5000, 3: 5000, 4: 5000, 5: 5000, 6: 5000, 7: 5000, 8: 5000, 9: 5000}
total: 50000


In [8]:
from pathlib import Path
from omegaconf import OmegaConf

cfg_dir = Path("/content/RAE/configs/stage2/training/ImageNet256_adapter")

for path in cfg_dir.glob("*.yaml"):
    cfg = OmegaConf.load(path)
    cfg.eval = None
    OmegaConf.save(cfg, path)
    print("patched", path)

patched /content/RAE/configs/stage2/training/ImageNet256_adapter/DiTDH-S_DINOv2-B_baseline.yaml
patched /content/RAE/configs/stage2/training/ImageNet256_adapter/DiTDH-S_DINOv2-B_mim4.yaml
patched /content/RAE/configs/stage2/training/ImageNet256_adapter/DiTDH-S_DINOv2-B_channel_se.yaml


In [15]:
import os, time

tag = time.strftime("%Y%m%d_%H%M%S")

os.environ["EXPERIMENT_NAME"] = f"dinov2_channel_se_cifar10_noeval_{tag}"
os.environ["ENTITY"] = "3297295031-"
os.environ["PROJECT"] = "rae-mim-dit"

os.environ["WANDB_ENTITY"] = os.environ["ENTITY"]
os.environ["WANDB_PROJECT"] = os.environ["PROJECT"]
os.environ["WANDB_RUN_NAME"] = os.environ["EXPERIMENT_NAME"]
os.environ["WANDB_RESUME"] = "never"

In [10]:
%cd /content/RAE

!python /content/RAE-DiT-Adapters/make_configs.py \
  --rae-root /content/RAE \
  --epochs 20 \
  --global-batch-size 64 \
  --num-workers 0

/content/RAE
baseline: /content/RAE/configs/stage2/training/ImageNet256_adapter/DiTDH-S_DINOv2-B_baseline.yaml
channel_se: /content/RAE/configs/stage2/training/ImageNet256_adapter/DiTDH-S_DINOv2-B_channel_se.yaml
mim4: /content/RAE/configs/stage2/training/ImageNet256_adapter/DiTDH-S_DINOv2-B_mim4.yaml


In [11]:
from pathlib import Path

p = Path("/content/RAE/src/utils/train_utils.py")
s = p.read_text()

if "from torch.cuda.amp import GradScaler" not in s:
    s = s.replace(
        "import torch\n",
        "import torch\nfrom torch.cuda.amp import GradScaler\n",
        1,
    )
    p.write_text(s)

!grep -n "GradScaler" /content/RAE/src/utils/train_utils.py

7:from torch.cuda.amp import GradScaler
100:def get_autocast_scaler(args) -> Tuple[dict, torch.cuda.amp.GradScaler | None]:
102:        scaler = GradScaler()


In [14]:
from pathlib import Path
from omegaconf import OmegaConf

cfg_path = Path("/content/RAE/configs/stage2/training/ImageNet256_adapter/DiTDH-S_DINOv2-B_channel_se.yaml")

cfg = OmegaConf.load(cfg_path)
print("before eval =", cfg.get("eval", None))

cfg.eval = None
OmegaConf.save(cfg, cfg_path)

cfg2 = OmegaConf.load(cfg_path)
print("after eval =", cfg2.get("eval", None))

before eval = {'eval_interval': 25000, 'eval_model': True, 'data_path': 'data/imagenet/val/', 'reference_npz_path': 'data/imagenet/VIRTUAL_imagenet256_labeled.npz'}
after eval = None


In [12]:
!rm -rf /content/RAE/wandb/run-20260606_035211-70979194

In [16]:
%cd /content/RAE

!torchrun --standalone --nnodes=1 --nproc_per_node=1 src/train.py \
  --config configs/stage2/training/ImageNet256_adapter/DiTDH-S_DINOv2-B_channel_se.yaml \
  --data-path /content/imagenet_like/train \
  --results-dir /content/rae_runs \
  --image-size 256 \
  --precision fp16 \
  --wandb \
  --compile

/content/RAE
2026-06-06 06:40:08.660317: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-06 06:40:08.730879: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/content/RAE/src/train.py:189: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_fp16)
[2026-06-06 06:40:15] Experiment directory created at /content/rae_runs/dinov2_channel_se_cifar10_noeval_20260606_063948
wandb: [wandb.login()] Loaded credentials 